# Day 1 · Session 2 — API Calls: Frontier & Open-Source Models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ksuresh/fdp-llm-agentic-ai/blob/main/day1-llm-foundations/02_api_calls_frontier_opensource.ipynb)

---

**Duration:** ~45 minutes  
**Goal:** Call 5 different AI providers using the same Python pattern. Build a multi-turn teaching assistant.

> *Adapted from Ed Donner's LLM Engineering course (week1/day3 & day4 + guides/09). Simplified and contextualised for faculty.*

---

## The Universal Pattern

```python
from openai import OpenAI
client = OpenAI(base_url="...", api_key="...")
response = client.chat.completions.create(
    model="...",
    messages=[{"role": "user", "content": "Your prompt"}]
)
print(response.choices[0].message.content)
```

**Change 3 lines → switch provider.** That's it.

In [ ]:
# Step 0 — Install (run once)
!pip install openai anthropic google-generativeai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import google.generativeai as genai

load_dotenv()

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
GOOGLE_API_KEY    = os.getenv("GOOGLE_API_KEY")

print("✅ Keys loaded")
print(f"   OpenAI:    {'✓' if OPENAI_API_KEY else '✗ NOT SET'}")
print(f"   Anthropic: {'✓' if ANTHROPIC_API_KEY else '✗ NOT SET'}")
print(f"   Google:    {'✓' if GOOGLE_API_KEY else '✗ NOT SET'}")

---

## Part 1 — OpenAI (GPT-4o-mini)

In [ ]:
# The test prompt — we'll use the same one across all providers
TEST_PROMPT = """
Explain Newton's Second Law (F = ma) in 3 sentences, 
with an example involving a cricket ball.
Keep it suitable for a first-year engineering student.
"""

# ── OpenAI ────────────────────────────────────────────────────
openai_client = OpenAI(api_key=OPENAI_API_KEY)

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": TEST_PROMPT}],
    max_tokens=300,
)

print("🟢 GPT-4o-mini:")
print(response.choices[0].message.content)
print(f"\n📊 Tokens used: {response.usage.total_tokens}")
# Cost estimate
cost_inr = (response.usage.prompt_tokens * 0.15 + response.usage.completion_tokens * 0.6) / 1_000_000 * 83.5
print(f"💰 Estimated cost: ₹{cost_inr:.5f}")

---

## Part 2 — Anthropic (Claude)

In [ ]:
# Anthropic has its own SDK (slightly different from OpenAI)
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

message = claude_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    messages=[{"role": "user", "content": TEST_PROMPT}]
)

print("🟠 Claude Haiku:")
print(message.content[0].text)
print(f"\n📊 Tokens: {message.usage.input_tokens} in + {message.usage.output_tokens} out")

In [ ]:
# Anthropic also works with the OpenAI-compatible client
# This is the unified pattern — same code, different base_url
claude_openai = OpenAI(
    base_url="https://api.anthropic.com/v1",
    api_key=ANTHROPIC_API_KEY,
    default_headers={"anthropic-version": "2023-06-01"}
)

response2 = claude_openai.chat.completions.create(
    model="claude-haiku-4-5",
    messages=[{"role": "user", "content": TEST_PROMPT}],
    max_tokens=300,
)

print("🟠 Claude Haiku (via OpenAI client):")
print(response2.choices[0].message.content)

---

## Part 3 — Google Gemini

In [ ]:
# Google uses its own SDK
genai.configure(api_key=GOOGLE_API_KEY)
gemini = genai.GenerativeModel("gemini-2.0-flash")

response3 = gemini.generate_content(TEST_PROMPT)

print("🔵 Gemini 2.0 Flash:")
print(response3.text)

---

## Part 4 — Ollama (Local, Free, Private)

> **Pre-requisite:** Ollama must be running locally.  
> Install from [ollama.com](https://ollama.com) then run: `ollama pull llama3.2`

Notice: **same OpenAI client, just different base_url and api_key**.

In [ ]:
# ── Ollama — local, free, no data leaves your machine ────────
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",   # required by the OpenAI client, but ignored by Ollama
)

response4 = ollama_client.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": TEST_PROMPT}],
    max_tokens=300,
)

print("⚫ LLaMA 3.2 (local via Ollama):")
print(response4.choices[0].message.content)
print("\n💰 Cost: ₹0.00 (running on your machine)")

In [ ]:
# See all models available in your local Ollama installation
import subprocess
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print("Available local models:")
print(result.stdout)

---

## Part 5 — DeepSeek (Cheap Cloud)

DeepSeek is ~10-50× cheaper than OpenAI. Uses the exact same OpenAI client pattern.

In [ ]:
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")

if DEEPSEEK_API_KEY:
    deepseek_client = OpenAI(
        base_url="https://api.deepseek.com",
        api_key=DEEPSEEK_API_KEY,
    )
    response5 = deepseek_client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": TEST_PROMPT}],
        max_tokens=300,
    )
    print("🔵 DeepSeek-V3:")
    print(response5.choices[0].message.content)
else:
    print("DEEPSEEK_API_KEY not set — skipping. Sign up free at platform.deepseek.com")

---

## Part 6 — Side-by-Side Comparison

In [ ]:
# Call all providers and display side by side
def call_model(provider, client, model, prompt, extra_headers=None):
    try:
        kwargs = {"model": model, "messages": [{"role": "user", "content": prompt}], "max_tokens": 200}
        response = client.chat.completions.create(**kwargs)
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

comparison_prompt = """
In exactly 2 sentences, what is the most important thing a
first-year engineering student should know about artificial intelligence?
"""

providers = [
    ("GPT-4o-mini",  openai_client,  "gpt-4o-mini"),
    ("Claude Haiku", claude_openai,  "claude-haiku-4-5"),
    ("LLaMA 3.2",    ollama_client,  "llama3.2"),
]

print(f"Prompt: {comparison_prompt.strip()}")
print("=" * 70)
for name, client_obj, model in providers:
    print(f"\n🔹 {name}:")
    print(call_model(name, client_obj, model, comparison_prompt))
    print("-" * 70)

---

## Part 7 — System Prompts

In [ ]:
# System prompt = instructor persona
# Without system prompt:
generic = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is backpropagation?"}
    ]
)

# With a well-crafted system prompt:
tailored = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": """
You are a teaching assistant for a Deep Learning course at an Indian engineering college.
Your students are in their 3rd year of B.Tech (CSE).
- Use analogies from everyday Indian life
- Include a small Python code snippet when relevant
- Keep explanations under 150 words
- End with one exam-style MCQ question on the topic
        """},
        {"role": "user", "content": "What is backpropagation?"}
    ]
)

print("WITHOUT system prompt:")
print(generic.choices[0].message.content[:400] + "...")
print("\n" + "=" * 60 + "\n")
print("WITH system prompt (Indian college context):")
print(tailored.choices[0].message.content)

---

## Part 8 — Streaming Output

In [ ]:
# Streaming — see the response appear token by token
# Great for student-facing apps — feels responsive even for long outputs
import sys

print("Streaming response from GPT-4o-mini:")
print("-" * 50)

stream = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "List 5 real-world applications of machine learning in Indian healthcare."}],
    stream=True,
    max_tokens=300,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

print("\n" + "-" * 50)

---

## Part 9 — Multi-Turn Conversation

In [ ]:
# Multi-turn conversation — maintain history
# This is how ChatGPT works: send the full conversation each time

SYSTEM_PROMPT = """
You are a Socratic teaching assistant for an AI/ML course at an Indian engineering college.
Instead of giving direct answers, guide students to discover the answer themselves
by asking probing questions. Keep responses under 80 words.
"""

conversation = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(user_message):
    conversation.append({"role": "user", "content": user_message})
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation,
        max_tokens=150,
    )
    assistant_msg = response.choices[0].message.content
    conversation.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg

# Simulate a student conversation
turns = [
    "I don't understand what a neural network is.",
    "Okay, so it's like neurons in a brain?",
    "But how does it learn from data?",
]

for turn in turns:
    print(f"👤 Student: {turn}")
    reply = chat(turn)
    print(f"🤖 Assistant: {reply}")
    print()

---

## 🎯 Your Turn — Lab Assignment

**File:** `lab/lab2_api_comparison.ipynb`

### Task 1
Call `gpt-4o-mini` with a question from your subject area. Print the response and the token count.

### Task 2  
Call `claude-haiku-4-5` with the **same question**. Compare the response style. What's different?

### Task 3
Run the same question locally with Ollama (`llama3.2`). Compare quality vs cost (₹0 vs cloud).

### Task 4
Build a multi-turn teaching assistant for your subject:
- Write a system prompt that defines the persona
- Have at least 3 turns of conversation
- The assistant should remember context from previous turns

### 🌟 Bonus
Add streaming output to your Task 4 assistant. Which model streams fastest?

```python
# Your code here
```